# Ternary-Bonsai-2-27B-Abliterated 一键部署

实例启动后，点击上方菜单 **Run All**（全部运行），等所有单元格执行完成即可。

本笔记本是幂等的：重复运行会跳过已下载的文件、不会重复启动服务；实例重启后直接重跑本笔记本即可。



In [ ]:
%%bash
echo "== ROCm 版本 =="
cat /opt/rocm/.info/version 2>/dev/null || echo "(未找到 /opt/rocm/.info/version)"
echo "== GPU =="
if command -v rocm-smi >/dev/null; then rocm-smi | head -20; elif command -v rocminfo >/dev/null; then rocminfo | head -20; else echo "(rocm-smi/rocminfo 不可用)"; fi
echo "== 磁盘 =="
df -h /workspace | tail -1



In [ ]:
%%bash
set -e
mkdir -p /workspace/prism-llama && cd /workspace/prism-llama
if find /workspace/prism-llama -name llama-server -type f | grep -q .; then
  echo "llama-server 已存在，跳过下载"
else
  URL=https://github.com/PrismML-Eng/llama.cpp/releases/download/prism-b10772-e8fd209/llama-prism-b10772-e8fd209-bin-ubuntu-rocm-7.2-x64.tar.gz
  if command -v wget >/dev/null; then
    wget -c --tries=3 --timeout=30 "$URL" || wget -c --tries=3 --timeout=30 "https://gh-proxy.com/$URL"
  else
    curl -L -C - --connect-timeout 20 --retry 3 -O "$URL" || curl -L -C - --connect-timeout 20 --retry 3 -O "https://gh-proxy.com/$URL"
  fi
  tar -xzf "$(basename "$URL")"
fi
BIN_DIR=$(find /workspace/prism-llama -name llama-server -type f | head -1 | xargs dirname)
echo "llama-server: $BIN_DIR/llama-server"
"$BIN_DIR/llama-server" --version



In [ ]:
%%bash
set -e
python3 -m pip install -U huggingface_hub -q || python3 -m pip install --break-system-packages -U huggingface_hub -q
export HF_ENDPOINT=https://hf-mirror.com
python3 - <<'PYEOF'
import os
from huggingface_hub import hf_hub_download
path = hf_hub_download(
    repo_id="Hikari07jp/Ternary-Bonsai-2-27B-Abliterated-GGUF",
    filename="Ternary-Bonsai-2-27B-Abliterated-PQ2_0.gguf",
    local_dir="/workspace/models/bonsai2-abliterated",
)
size = os.path.getsize(path)
print(f"下载完成: {path} ({size / 1e9:.2f} GB)")
assert size == 7206168928, "文件大小不符，请删除后重跑本单元格"
PYEOF



In [ ]:
%%bash
set -e
BIN_DIR=$(find /workspace/prism-llama -name llama-server -type f | head -1 | xargs dirname)
if curl -sf http://127.0.0.1:8080/v1/models >/dev/null 2>&1; then
  echo "llama-server 已在运行"
else
  nohup "$BIN_DIR/llama-server" \
    -m /workspace/models/bonsai2-abliterated/Ternary-Bonsai-2-27B-Abliterated-PQ2_0.gguf \
    -ngl 999 -fa on -c 32768 \
    --host 127.0.0.1 --port 8080 \
    --alias bonsai2-abliterated \
    --jinja \
    --temp 1.0 --top-p 0.95 --top-k 20 \
    --chat-template-kwargs '{"reasoning_effort":"medium"}' \
    > /workspace/llama-server.log 2>&1 &
  echo "已启动，等待模型加载（首次约 30-90 秒）..."
fi
for i in $(seq 1 60); do
  curl -sf http://127.0.0.1:8080/v1/models >/dev/null 2>&1 && { echo "服务已就绪"; break; }
  sleep 5
done
tail -5 /workspace/llama-server.log



In [ ]:
%%bash
echo "== 模型列表 =="
curl -s http://127.0.0.1:8080/v1/models; echo
echo "== 对话测试 =="
curl -s http://127.0.0.1:8080/v1/chat/completions -H 'Content-Type: application/json' -d '{"model":"bonsai2-abliterated","messages":[{"role":"user","content":"用一句话介绍你自己"}],"max_tokens":128}'; echo
echo "== 下一步：在 Windows 本机 Git Bash 里开隧道（保持窗口开着）=="
echo 'ssh -p <平台端口> <用户名>@<平台地址> -L 8080:127.0.0.1:8080 -N'



## 完成后（在 Windows 本机操作）

1. Git Bash 开 SSH 隧道（保持窗口不关）：
   ssh -p <平台端口> <用户名>@<平台地址> -L 8080:127.0.0.1:8080 -N
2. 验证：本机浏览器打开 http://127.0.0.1:8080 ，能直接和模型对话即成功。
3. Qoder 设置 → 模型 → 添加 → 自定义 → OpenAI Compatible（Chat Completions API）：
   - Base URL: http://127.0.0.1:8080/v1
   - API Key: sk-local（任意字符串）
   - Model ID: bonsai2-abliterated

实例重启后：重跑本笔记本即可（已下载内容会自动跳过）。

